In [ ]:
import numpy as np

NUM_PHASES = 4
G_MIN = 15.0
G_MAX = 60.0

ARRIVAL_RATES = np.array([0.25, 0.40, 0.15, 0.30])
INITIAL_QUEUES = np.array([12, 25, 8, 18])
SATURATION_FLOW = 0.5

def objective_function(green_times):
    """
    Evaluates a traffic signal timing plan (a wolf's position).
    Returns a congestion index based on waiting time and remaining queue lengths.
    Goal: Minimize this value.
    """
    w1, w2 = 0.6, 0.4
    total_waiting_time = 0
    remaining_queues = 0

    for i in range(NUM_PHASES):
        g = green_times[i]
        arrival = ARRIVAL_RATES[i]
        q_init = INITIAL_QUEUES[i]

        discharged = min(q_init + (arrival * g), SATURATION_FLOW * g)
        q_end = max(0.0, q_init + (arrival * g) - discharged)

        avg_queue = (q_init + q_end) / 2.0
        waiting_time = avg_queue * g

        total_waiting_time += waiting_time
        remaining_queues += q_end

    return (w1 * total_waiting_time) + (w2 * remaining_queues)

def grey_wolf_optimizer(num_wolves=30, max_iter=100):

    positions = np.random.uniform(G_MIN, G_MAX, (num_wolves, NUM_PHASES))

    alpha_pos = np.zeros(NUM_PHASES)
    beta_pos = np.zeros(NUM_PHASES)
    delta_pos = np.zeros(NUM_PHASES)

    alpha_score = float("inf")
    beta_score = float("inf")
    delta_score = float("inf")

    print("Running Grey Wolf Optimizer for Traffic Intersections...\n")

    for l in range(max_iter):

        for i in range(num_wolves):

            positions[i] = np.clip(positions[i], G_MIN, G_MAX)
            fitness = objective_function(positions[i])

            if fitness < alpha_score:
                alpha_score = fitness
                alpha_pos = positions[i].copy()

            elif fitness < beta_score:
                beta_score = fitness
                beta_pos = positions[i].copy()

            elif fitness < delta_score:
                delta_score = fitness
                delta_pos = positions[i].copy()

        a = 2 - l * (2 / max_iter)

        for i in range(num_wolves):

            r1, r2 = np.random.rand(NUM_PHASES), np.random.rand(NUM_PHASES)
            A1 = 2 * a * r1 - a
            C1 = 2 * r2
            D_alpha = np.abs(C1 * alpha_pos - positions[i])
            X1 = alpha_pos - A1 * D_alpha

            r1, r2 = np.random.rand(NUM_PHASES), np.random.rand(NUM_PHASES)
            A2 = 2 * a * r1 - a
            C2 = 2 * r2
            D_beta = np.abs(C2 * beta_pos - positions[i])
            X2 = beta_pos - A2 * D_beta

            r1, r2 = np.random.rand(NUM_PHASES), np.random.rand(NUM_PHASES)
            A3 = 2 * a * r1 - a
            C3 = 2 * r2
            D_delta = np.abs(C3 * delta_pos - positions[i])
            X3 = delta_pos - A3 * D_delta

            positions[i] = (X1 + X2 + X3) / 3.0

        if (l + 1) % 20 == 0 or l == 0:
            print(f"Iteration {l+1:03d} | Minimum Congestion Index (Alpha): {alpha_score:.4f}")

    return alpha_pos, alpha_score

if __name__ == "__main__":
    best_timing, best_congestion = grey_wolf_optimizer(num_wolves=30, max_iter=100)

    print("\n==============================================")
    print("🎯 OPTIMIZED TRAFFIC SIGNAL TIMES (GWO)")
    print("==============================================")
    for phase, seconds in enumerate(best_timing, start=1):
        print(f"  • Phase {phase} Green Time: {seconds:.2f} seconds")
    print(f"Minimized Gridlock Metric Score: {best_congestion:.4f}")


Running Grey Wolf Optimizer for Traffic Intersections...

Iteration 001 | Minimum Congestion Index (Alpha): 679.6506
Iteration 020 | Minimum Congestion Index (Alpha): 526.0500
Iteration 040 | Minimum Congestion Index (Alpha): 526.0500
Iteration 060 | Minimum Congestion Index (Alpha): 526.0500
Iteration 080 | Minimum Congestion Index (Alpha): 526.0500
Iteration 100 | Minimum Congestion Index (Alpha): 526.0500

🎯 OPTIMIZED TRAFFIC SIGNAL TIMES (GWO)
  • Phase 1 Green Time: 15.00 seconds
  • Phase 2 Green Time: 15.00 seconds
  • Phase 3 Green Time: 15.00 seconds
  • Phase 4 Green Time: 15.00 seconds
Minimized Gridlock Metric Score: 526.0500
